# Model Comparison — All Results
Merges `classical_results.json` + `indobert_results.json` into one summary.  
**Run after** both `01_classical_models.ipynb` and `02_indobert_finetune.ipynb` have completed.

---
## 1. Imports

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import ConfusionMatrixDisplay
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "/mnt/user-data/outputs/final_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Imports OK")

## 2. Load Results

In [ ]:
with open("./outputs/classical_models/classical_results.json") as f: ...
with open("./outputs/indobert_model/indobert_results.json") as f: ...

OUTPUT_DIR = "../outputs/final_results"

print(f"Classical models loaded : {[r['model'] for r in classical]}")
print(f"IndoBERT loaded         : {indobert['model']}")

## 3. Unified Comparison Table

In [ ]:
rows = []
for r in classical:
    rows.append({
        "Model":      r["model"],
        "Accuracy":   r["accuracy"],
        "Precision":  r["precision_macro"],
        "Recall":     r["recall_macro"],
        "F1":         r["f1_macro"],
        "AUC-ROC":    r["auc_roc"],
        "ms/sample":  r["inference_ms_per_sample"],
    })

rows.append({
    "Model":     indobert["model"],
    "Accuracy":  indobert["test_accuracy"],
    "Precision": indobert["test_precision_macro"],
    "Recall":    indobert["test_recall_macro"],
    "F1":        indobert["test_f1_macro"],
    "AUC-ROC":   indobert["test_auc_roc"],
    "ms/sample": indobert["inference_ms_per_sample"],
})

rows.sort(key=lambda x: x["F1"], reverse=True)

print(f"{'Model':<35} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7} {'ms/s':>8}")
print("─" * 80)
for r in rows:
    acc_flag = " ✓" if r["Accuracy"] >= 0.85 else "  "
    f1_flag  = " ✓" if r["F1"]       >= 0.80 else "  "
    auc = f"{r['AUC-ROC']:.4f}" if r["AUC-ROC"] else "  N/A"
    print(f"{r['Model']:<35} {r['Accuracy']:>7.4f}{acc_flag} "
          f"{r['Precision']:>7.4f} {r['Recall']:>7.4f} "
          f"{r['F1']:>7.4f}{f1_flag} {auc:>7} {r['ms/sample']:>8.4f}")
print()
print("✓ = meets target  |  Accuracy ≥ 0.85  ·  F1 ≥ 0.80")

## 4. Bar Chart — Metrics Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model Comparison — Test Set Metrics", fontsize=14, fontweight="bold")

models     = [r["Model"].replace(" (LinearSVC + Calibrated)", "") for r in rows]
short_names = [m.replace("IndoBERT (indobert-base-p2)", "IndoBERT")
                .replace("Multinomial Naive Bayes", "MNB")
                .replace("Logistic Regression", "LR")
                .replace("SVM", "SVM") for m in models]

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"][:len(rows)]
x = np.arange(len(rows))
w = 0.2

# Left: Accuracy, Precision, Recall, F1
metrics_left = ["Accuracy", "Precision", "Recall", "F1"]
for i, metric in enumerate(metrics_left):
    vals = [r[metric] for r in rows]
    axes[0].bar(x + i * w, vals, width=w, label=metric, color=colors[i], alpha=0.85)

axes[0].set_xticks(x + w * 1.5)
axes[0].set_xticklabels(short_names, fontsize=10)
axes[0].set_ylim(0.7, 1.0)
axes[0].set_ylabel("Score")
axes[0].set_title("Accuracy / Precision / Recall / F1")
axes[0].legend(fontsize=9)
axes[0].axhline(0.85, color="red",  linestyle="--", linewidth=1, alpha=0.6, label="Acc target")
axes[0].axhline(0.80, color="blue", linestyle=":",  linewidth=1, alpha=0.6, label="F1 target")
axes[0].legend(fontsize=9)

# Right: AUC-ROC
auc_vals = [r["AUC-ROC"] if r["AUC-ROC"] else 0 for r in rows]
bars = axes[1].bar(short_names, auc_vals, color=colors, alpha=0.85, width=0.5)
axes[1].set_ylim(0.7, 1.0)
axes[1].set_ylabel("AUC-ROC")
axes[1].set_title("AUC-ROC")
for bar, val in zip(bars, auc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                  f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/metrics_comparison.png")

## 5. IndoBERT Training Curve

In [ ]:
history = indobert["training_history"]
epochs     = [h["epoch"]      for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss   = [h["val_loss"]   for h in history]
val_f1     = [h["val_f1"]     for h in history]
val_acc    = [h["val_accuracy"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("IndoBERT Training History", fontsize=13, fontweight="bold")

ax1.plot(epochs, train_loss, "o-", color="#4C72B0", label="Train loss")
ax1.plot(epochs, val_loss,   "s--", color="#DD8452", label="Val loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss"); ax1.legend()
ax1.set_xticks(epochs)

ax2.plot(epochs, val_f1,  "o-",  color="#55A868", label="Val F1")
ax2.plot(epochs, val_acc, "s--", color="#C44E52", label="Val Accuracy")
ax2.axhline(0.85, color="red",  linestyle="--", linewidth=1, alpha=0.5, label="Acc target 0.85")
ax2.axhline(0.80, color="green", linestyle=":", linewidth=1, alpha=0.5, label="F1 target 0.80")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("Val F1 & Accuracy"); ax2.legend(fontsize=8)
ax2.set_xticks(epochs)
# Mark best epoch
best_ep = indobert["best_epoch"]
ax2.axvline(best_ep, color="gray", linestyle=":", alpha=0.7)
ax2.text(best_ep + 0.05, min(val_f1) + 0.005, f"best (ep {best_ep})", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/indobert_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/indobert_training_curve.png")

## 6. Confusion Matrices (All Models)

In [ ]:
all_cms    = [r["confusion_matrix"] for r in classical] + [indobert["confusion_matrix"]]
all_names  = [r["model"] for r in classical] + [indobert["model"]]
short_plot = ["MNB", "Logistic Regression", "SVM", "IndoBERT"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Confusion Matrices — Test Set", fontsize=13, fontweight="bold")

for ax, cm, name in zip(axes, all_cms, short_plot):
    cm_arr = np.array(cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_arr,
                                   display_labels=["Non-CB", "CB"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=11)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/confusion_matrices.png")

## 7. Inference Speed Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
names  = [r["Model"].replace("IndoBERT (indobert-base-p2)", "IndoBERT")
                     .replace(" (LinearSVC + Calibrated)", "")
                     .replace("Multinomial Naive Bayes", "MNB") for r in rows]
speeds = [r["ms/sample"] for r in rows]
colors_spd = ["#4C72B0"] * (len(rows) - 1) + ["#DD8452"]

bars = ax.barh(names, speeds, color=colors_spd, alpha=0.85, height=0.5)
ax.set_xlabel("Inference time (ms per sample) — lower is better")
ax.set_title("Inference Speed Comparison")

for bar, val in zip(bars, speeds):
    ax.text(bar.get_width() + max(speeds) * 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.4f} ms", va="center", fontsize=9)
ax.set_xlim(0, max(speeds) * 1.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/inference_speed.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/inference_speed.png")

## 8. Save Combined Results JSON

In [ ]:
combined = {
    "summary_table": rows,
    "targets": {"accuracy": 0.85, "f1_macro": 0.80},
    "classical_detail": classical,
    "indobert_detail": indobert,
}
with open(f"{OUTPUT_DIR}/all_results.json", "w") as f:
    json.dump(combined, f, indent=2)

print(f"Saved: {OUTPUT_DIR}/all_results.json")
print()
print("All outputs in:", OUTPUT_DIR)
for fname in os.listdir(OUTPUT_DIR):
    print(f"  {fname}")